In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
import pandas as pd
from PycalcAct.dataset import Dataset
from PycalcAct.model import (
    MixedFCTemporalModel)
from PycalcAct.trainer import Trainer
from PycalcAct.calculateCustomAccuracy import calculateCustomAccuracy
from pathlib import Path
from PycalcAct.myCustomCriterion import myCustomCriterion
_ = torch.manual_seed(1234)


dataFolder = "//Hmr_lymph/d/SebastienThis/CalciumPredictions/Ca2-Analysis_McGill/prediction/agAffinity/datasets/trainingData/"

csv_path=[dataFolder + "legend.csv", dataFolder+"calciumRatio_normalized.csv"] 
csv_pos_path = dataFolder + "position.csv"

dataset = Dataset(
    csv_path = csv_path,
    csv_pos_path=csv_pos_path,  # Optional
    position_to_displacement=True,
    remove_mean=True,
    replace_nan_by_min = True,
    customFilter = False,
    is_regression = False, #True
)
dataset.create_new_feature_by_operations(lambda x: np.abs(np.fft.fft(x[:, 0])))

# Setup model
model = MixedFCTemporalModel(
    n_classes=4, #1,
    n_rnn_layers=3,
    rnn_hidden_size=32,
    n_fc_layers=1,
    fc_hidden_size=32,
    temporal_length=dataset.length_serie,
    input_size=dataset.features,
    bidirectional=True,
    pooling=None,
    dropout=0.2
)   

# Setup training
n_epochs = 500
        
trainer = Trainer(
    dataset,
    model,
    device="cuda",
    batch_size=2000,
    criterion=None,
    is_regression=False, #True,
    regression_bounds = None,#torch.Tensor([-12,-10,-9]).to("cuda")
)  # You can pass your own optimizer, criterion, learning rate, weight decay and learning rate scheduler.

#trainer.summary()   
print(trainer.dataset.y_train)

[3 3 0 ... 1 3 1]


In [2]:
trainer.train(n_epochs)

╭───────────────────────────┬───────────────────────────┬───────────────────────────╮
│           Epoch           │                      Loss │                  Accuracy │
├───────────────────────────┼───────────────────────────┼───────────────────────────┤
│             0             │                      1.36 │                     32.55 │
│             1             │                      1.32 │                     37.91 │
│             2             │                      1.32 │                     40.89 │
│             4             │                      1.24 │                     41.87 │
│             8             │                      1.25 │                     44.93 │
│             10            │                      1.20 │                     46.47 │
│             14            │                      1.17 │                     49.19 │
│             17            │                      1.14 │                     49.24 │
│             18            │                      1.1

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
import pandas as pd
from PycalcAct.dataset import Dataset
from PycalcAct.model import (
    MixedFCTemporalModel)
from PycalcAct.trainer import Trainer
from PycalcAct.calculateCustomAccuracy import calculateCustomAccuracy
from pathlib import Path
from PycalcAct.myCustomCriterion import myCustomCriterion
_ = torch.manual_seed(1234)


dataFolder = "//Hmr_lymph/d/SebastienThis/CalciumPredictions/Ca2-Analysis_McGill/prediction/agAffinity/datasets/trainingData/"

csv_path=[dataFolder + "legend.csv", dataFolder+"calciumRatio_normalized.csv"] 
csv_pos_path = dataFolder + "position.csv"

dataset = Dataset(
    csv_path = csv_path,
    csv_pos_path=csv_pos_path,  # Optional
    position_to_displacement=True,
    remove_mean=True,
    replace_nan_by_min = True,
    customFilter = False,
    is_regression = True
)
dataset.create_new_feature_by_operations(lambda x: np.abs(np.fft.fft(x[:, 0])))

# Setup model
model = MixedFCTemporalModel(
    n_classes= 1,
    n_rnn_layers=3,
    rnn_hidden_size=32,
    n_fc_layers=1,
    fc_hidden_size=32,
    temporal_length=dataset.length_serie,
    input_size=dataset.features,
    bidirectional=True,
    pooling=None,
    dropout=0.2
)   

# Setup training
n_epochs = 500
        
trainer = Trainer(
    dataset,
    model,
    
    device="cuda",
    batch_size=2000,
    criterion=None,
    is_regression=True,
    regression_bounds = torch.Tensor([0.5,1.5,2.5]).to("cuda") #torch.Tensor([-12,-10,-9]).to("cuda")
)  # You can pass your own optimizer, criterion, learning rate, weight decay and learning rate scheduler.

#trainer.summary()   
print(trainer.dataset.y_train)

[1 0 2 ... 3 1 0]


In [4]:
trainer.train(n_epochs)

d:\Sebastien\Anaconda\envs\calcium\lib\site-packages\torch\nn\modules\loss.py:610: UserWarning: Using a target size (torch.Size([2000])) that is different to the input size (torch.Size([2000, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)
d:\Sebastien\Anaconda\envs\calcium\lib\site-packages\torch\nn\modules\loss.py:610: UserWarning: Using a target size (torch.Size([684])) that is different to the input size (torch.Size([684, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)


╭───────────────────────────┬───────────────────────────┬───────────────────────────╮
│           Epoch           │                      Loss │                  Accuracy │
├───────────────────────────┼───────────────────────────┼───────────────────────────┤
│             0             │                      1.13 │                     31.19 │
│            499            │                      0.99 │                           │
╰───────────────────────────┴───────────────────────────┴───────────────────────────╯


In [ ]:
from torch import DoubleTensor
x = torch.tensor(trainer.dataset.x_test).to("cuda")
print(x)
y_pred = trainer.model(x.type(torch.float32))

tensor([[[-6.4691e-01, -6.4691e-01, -6.4691e-01,  ..., -4.4851e-01,
          -6.4691e-01, -6.1481e-01],
         [ 0.0000e+00,  0.0000e+00,  0.0000e+00,  ...,  6.0855e-01,
           1.9288e+00,  4.5057e+00]],

        [[-2.0512e+00, -2.0433e+00,  3.4455e+00,  ..., -2.6041e-01,
          -5.3894e-01, -5.5832e-01],
         [ 2.1551e+03,  1.0076e+01,  1.3657e+01,  ...,  5.4676e+00,
           1.9076e+00,  9.9287e-01]],

        [[-1.3088e+00, -1.0955e+00, -1.2102e+00,  ..., -1.3088e+00,
          -1.3088e+00, -1.3088e+00],
         [ 0.0000e+00,  1.7077e+03,  1.4131e+01,  ...,  0.0000e+00,
           0.0000e+00,  0.0000e+00]],

        ...,

        [[-1.9032e+00, -1.9032e+00, -1.9032e+00,  ..., -1.9032e+00,
          -1.9032e+00, -1.9032e+00],
         [ 0.0000e+00,  0.0000e+00,  0.0000e+00,  ...,  0.0000e+00,
           0.0000e+00,  0.0000e+00]],

        [[-8.3904e-01, -8.3904e-01, -8.3904e-01,  ..., -5.7090e-01,
          -2.3231e-01, -2.3804e-01],
         [ 0.0000e+00,  0.0000e+0

ValueError: input must have the type torch.float32, got type torch.float64

In [5]:
# train model

# from progress_table import ProgressTable
# from copy import deepcopy
# from torchmetrics.classification import Accuracy, CohenKappa, ConfusionMatrix

# from torch import tensor
# target = tensor([1, 1, 0, 0])
# preds = tensor([0, 1, 0, 0])
# cohenkappa = CohenKappa(task="multiclass", num_classes=2)
# cohenkappa(preds, target)

# if trainer.scheduler is None:
#     trainer.scheduler = trainer.default_scheduler()(T_max=n_epochs)
#     trainer._initial_scheduler_state_dict = deepcopy(trainer.scheduler.state_dict())
#     trainer._initial_scheduler_state_dict = deepcopy(trainer.scheduler.state_dict())

# x, y = trainer.dataset.train_batch(True, to_cuda=True)
# batch_size = len(x) if trainer.batch_size is None else trainer.batch_size

# current_best = 0
# xval, yval = trainer.dataset.val_batch(True, to_cuda=True)
# table = ProgressTable(
#     num_decimal_places=2,
#     print_header_every_n_rows=15,
#     # print_header_every_n_rows=15,
#     pbar_show_progress=True,
#     pbar_style="square",
#     default_column_width=25,
# )

# table.add_column("Epoch")
# table.add_column("Loss", color="blue", alignment="right")
# table.add_column(trainer._store_best, color="green", alignment="right")
# table.add_column("Loss", color="blue", alignment="right")
# table.add_column(trainer._store_best, color="green", alignment="right")

# for e in table(
#     range(n_epochs),
#     total=n_epochs,
#     description="Epoch",
#     show_eta=True,
# ):
#     idx = torch.randperm(x.shape[0], device=x.device)
#     x = x[idx]
#     y = y[idx]

#     trainer.model.train()
#     i = 1
#     # for i in range(0, len(x), batch_size):
#     trainer.optim.zero_grad()
#     x_batch = x[i : i + batch_size]
#     y_batch = y[i : i + batch_size].type(torch.FloatTensor).to(trainer.device)
#         # Take batch size:
#     with torch.autocast("cuda"): #torch.cuda.amp.autocast()
#         y_pred = trainer.model(x_batch)
#         loss = trainer.criterion(y_pred.squeeze(), y_batch)
#         loss.backward()
#         trainer.optim.step()

#     if trainer.scheduler:
#         trainer.scheduler.step()
#     table.update("Loss", loss.item(), aggregate="mean", color="blue")
#     table.update("Epoch", e)
#     table.update("Loss", loss.item(), aggregate="mean", color="blue")
#     table.update("Epoch", e)
#     if e % 1 == 0:
#         trainer.model.eval()
#         trainer.metrics.reset()
#         trainer.confmat.reset()

#         batch_size = len(x) if trainer.batch_size is None else trainer.batch_size

#         with torch.no_grad():
#             for i in range(0, len(x), batch_size):
#                 xbatch = x[i : i + batch_size]
#                 if trainer.is_regression:
#                     ybatch = y[i : i + batch_size].type(torch.FloatTensor).to(trainer.device)
#                 else:
#                     ybatch = y[i : i + batch_size].type(torch.LongTensor).to(trainer.device)

#                 y_pred = trainer.model(xbatch)
#                 loss = trainer.criterion(y_pred.squeeze(), ybatch)

#                 if trainer.is_regression:
#                     # From continuous to categorical
#                     # use regression_bounds to define the thresholds
#                     y_pred = torch.bucketize(y_pred.squeeze(), trainer.regression_bounds).to(trainer.device)
#                     ybatch = torch.bucketize(ybatch.squeeze(), trainer.regression_bounds).to(trainer.device)
#                 else:
#                     y_pred = torch.softmax(y_pred, dim=1)
#                 trainer.metrics.update(y_pred, ybatch)
#                 trainer.confmat.update(y_pred, ybatch)
#                 scores = trainer.metrics.compute()

#                 if scores[trainer._store_best] > current_best:
#                     current_best = scores[trainer._store_best]
#                     trainer._best_state_dict = deepcopy(trainer.model.state_dict())
#                     table.update(trainer._store_best, scores[trainer._store_best].item() * 100, color="green")
#                     table.update(trainer._store_best, scores[trainer._store_best].item() * 100, color="green")

#                     table.next_row()


In [6]:
#         # save metrics
# trainer.register_last_state()
# fig, axs = plt.subplots(1, 3, figsize=(18, 5))

# trainer.model.load_state_dict(trainer._best_state_dict)
# fig.suptitle("Best Model")
# fig.suptitle("Best Model")


# callbacks = (
#         trainer.dataset.train_batch,
#         trainer.dataset.val_batch,
#         trainer.dataset.test_batch,
# )

# for i, (name, callable) in enumerate(zip(["Train", "Val", "Test"], callbacks)):
#     x, y = callable(True, to_cuda=True)
#     trainer.model.eval()
#     trainer.metrics.reset()
#     trainer.confmat.reset()

#     batch_size = len(x) if trainer.batch_size is None else trainer.batch_size

#     with torch.no_grad():
#         for i in range(0, len(x), batch_size):
#             xbatch = x[i : i + batch_size]
#             if trainer.is_regression:
#                 ybatch = y[i : i + batch_size].type(torch.FloatTensor).to(trainer.device)
#             else:
#                 ybatch = y[i : i + batch_size].type(torch.LongTensor).to(trainer.device)

#             y_pred = trainer.model(xbatch)
#             loss = trainer.criterion(y_pred.squeeze(), ybatch)

#             if trainer.is_regression:
#                 # From continuous to categorical
#                 # use regression_bounds to define the thresholds
#                 y_pred = torch.bucketize(y_pred.squeeze(), trainer.regression_bounds).to(trainer.device)
#                 ybatch = torch.bucketize(ybatch.squeeze(), trainer.regression_bounds).to(trainer.device)
#             else:
#                 y_pred = torch.softmax(y_pred, dim=1)
#             trainer.metrics.update(y_pred, ybatch)
#             trainer.confmat.update(y_pred, ybatch)
#             scores = trainer.metrics.compute()
#     axs[i].set_title(
#     f"Accuracy {(name)} {scores['Accuracy'].item():.2%}, Cohen's Kappa {scores['CohenKappa'].item():.1%}"
#     )
#     trainer.confmat.plot(
#     ax=axs[i],
#     cmap="RdYlGn",
#     labels=trainer.dataset.labels,
#     )
# fig.show()
   
# # narr = np.array([metrics['Accuracy'].tolist(), metrics['CohenKappa'].tolist()])
# fig.show()
